<a href="https://colab.research.google.com/github/hourglassfigure-smartman/Python-basic-kadai/blob/main/automate_ordering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import datetime
import smtplib
from email.mime.text import MIMEText
from google.colab import drive
import os

# 1. Googleドライブをマウント
drive.mount('/content/drive')

# パスの設定 (マイドライブ直下に samples フォルダがある想定)
base_path = '/content/drive/MyDrive/samples/'

# 複数の注文ファイルの名前をリストで定義
order_files_list = [
    os.path.join(base_path, 'order_new/order_A_20230524.xlsx'),
    os.path.join(base_path, 'order_new/order_B_20230524.xlsx'),
    os.path.join(base_path, 'order_new/order_C_20230524.xlsx'),
    os.path.join(base_path, 'order_new/order_D_20230524.xlsx')
]
inventory_file = os.path.join(base_path, 'inventory.xlsx')
pickup_file = os.path.join(base_path, 'pickup.xlsx')

# --- 開発工程 ---

# 2. 注文情報をまとめる [cite: 70, 71]
all_orders_df = pd.DataFrame() # 全注文を格納する空のDataFrameを初期化

for file_path in order_files_list:
    df = pd.read_excel(file_path)
    all_orders_df = pd.concat([all_orders_df, df], ignore_index=True)

# DataFrameの各列（商品）の合計を計算
summary_orders = all_orders_df.sum()
print("各野菜の合計注文数:")
print(summary_orders)


# 3. 現在の在庫状況を確認する [cite: 72, 73]
df_inventory = pd.read_excel(inventory_file)
latest_inventory = df_inventory.iloc[-1]  # 最終行を抽出

# 4. 発注が必要なアイテムを特定する [cite: 74, 75, 77, 78]
df_pickup = pd.read_excel(pickup_file)
order_list = []

# Extract item names (excluding the first column, 'Unnamed: 0')
item_columns = df_pickup.columns[1:]

for item_name in item_columns:
    # Get threshold for the current item
    threshold = df_pickup.loc[df_pickup['Unnamed: 0'] == 'しきい値', item_name].iloc[0]
    # Get order amount for the current item
    order_amount = df_pickup.loc[df_pickup['Unnamed: 0'] == '追加量', item_name].iloc[0]

    # 最新在庫 - 合計注文数 = 現在の理論在庫
    current_stock = latest_inventory.get(item_name, 0) - summary_orders.get(item_name, 0)

    # しきい値を下回った場合 [cite: 78]
    if current_stock < threshold:
        order_list.append({'商品名': item_name, '数量': order_amount})

# 5. 発注メールの作成と送信 [cite: 79, 80, 81, 82]
if order_list:
    # メール本文の作成
    mail_body = "農家様\n\nお疲れ様です。以下の野菜を発注いたします。\n\n"
    for item in order_list:
        mail_body += f"・{item['商品名']}: {item['数量']}個\n"
    mail_body += "\nよろしくお願いいたします。"

    # Mailtrap経由で送信 (機密情報は伏字にしています)
    username = '910892537401fa' # Mailtrapのユーザー名
    password = '533c56ffddef1f' # Mailtrapのパスワード
    sender_address = '*******************'
    receiver_address = '*******************'

    msg = MIMEText(mail_body)
    msg['Subject'] = '野菜の発注依頼'
    msg['From'] = sender_address
    msg['To'] = receiver_address

    with smtplib.SMTP('sandbox.smtp.mailtrap.io', 2525) as server:
        server.login(username, password)
        server.send_message(msg)
    print("発注メールを送信しました。")

# 6. 在庫状況を最新の情報に更新する [cite: 87, 89, 90, 91]
# 新しい行のデータ作成
new_inventory_data = {'日付': datetime.date.today()}
for item in df_inventory.columns:
    if item == '曜日':
        new_inventory_data[item] = datetime.date.today().strftime('%a') # 現在の曜日を設定
    elif item != '日付':
        # 在庫更新: 最新在庫 - 注文数 (発注分は届く前なので含めない)
        new_val = latest_inventory.get(item, 0) - summary_orders.get(item, 0)
        new_inventory_data[item] = new_val

# データの追加と保存
df_new_row = pd.DataFrame([new_inventory_data])
df_inventory = pd.concat([df_inventory, df_new_row], ignore_index=True)
df_inventory.to_excel(inventory_file, index=False)
print("在庫表を更新しました。")

print("--- 生成されたメール本文 ---")
print(mail_body)

print("--- 計算された最新在庫から注文数を引いた結果 ---")
# '日付'と'曜日'は数値計算の対象外なので除外して表示
# new_inventory_dataには、すでに'最新在庫 - 注文数'の結果が格納されています。
# また、new_inventory_dataの辞書はPythonのプリミティブ型で、DataFrameへの変換は次セルで行われるため、
# ここではPandas Seriesとして表示することで、より見やすくします。
calculated_remaining_stock = pd.Series({
    item: value for item, value in new_inventory_data.items()
    if item not in ['日付', '曜日']
})
print(calculated_remaining_stock)

print("\n--- 閾値を下回った品目（発注リスト）---")
print(order_list)

ValueError: mount failed